In [11]:
%pip install requests beautifulsoup4 pandas spacy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from bs4 import BeautifulSoup
import spacy
import re
import json
import pandas as pd
import unicodedata

url = "https://en.wikipedia.org/wiki/Steve_Jobs"
html = requests.get(url, headers={"User-Agent": "StudentBot/1.0"}).text
soup = BeautifulSoup(html, "html.parser")

article = soup.find("div", id="mw-content-text")
for tag in article.select("table, sup, .mw-editsection, .navbox"):
    tag.decompose()

text = " ".join(p.get_text(" ", strip=True)
                for p in article.find_all("p"))

# Preprocess and normalize
text = unicodedata.normalize("NFKC", text)
text = re.sub(r"\[\d+\]", "", text)
text = re.sub(r"\s+", " ", text).strip()


# Extract named entities
nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

entities = []
for ent in doc.ents:
    if ent.label_ in ["PERSON", "ORG", "GPE", "DATE", "WORK_OF_ART"]:
        entities.append({
            "entity": ent.text,
            "type": ent.label_
        })

# Extract years and decades using regex
for date in set(re.findall(r"\b(?:19|20)\d{2}s?\b", text)):
    entities.append({"entity": date, "type": "DATE"})

# Remove duplicates and create a table
entities = [dict(t) for t in {
    (e["entity"], e["type"]): e for e in entities
}.values()]


# Create a DataFrame of all entities
df = pd.DataFrame(entities)

# Categorize entities by type
persons = df[df["type"] == "PERSON"]
organizations = df[df["type"] == "ORG"]
locations = df[df["type"] == "GPE"]
dates = df[df["type"] == "DATE"]
works = df[df["type"] == "WORK_OF_ART"]

# Display each category
print("PERSONS")
print(persons)

print("\nORGANIZATIONS")
print(organizations)

print("\nLOCATIONS")
print(locations)

print("\nDATES")
print(dates)

print("\n WORKS / BOOK TITLES")
print(works)

# Save entities to JSON
with open("steve_jobs_entities.json", "w", encoding="utf-8") as f:
    json.dump(entities, f, ensure_ascii=False, indent=4)

print("\nEntities saved to steve_jobs_entities.json")

PERSONS
                 entity    type
0      Steven Paul Jobs  PERSON
5                  Jobs  PERSON
7         Steve Wozniak  PERSON
27              Wozniak  PERSON
39         John Sculley  PERSON
..                  ...     ...
579      Warren Buffett  PERSON
580  Walter Isaacson 's  PERSON
582        Aaron Sorkin  PERSON
583  Michael Fassbender  PERSON
586         Alex Gibney  PERSON

[155 rows x 2 columns]

ORGANIZATIONS
                                entity type
6                           Apple Inc.  ORG
8               Apple Computer Company  ORG
11                                NeXT  ORG
13                               Pixar  ORG
16                               Apple  ORG
..                                 ...  ...
565                            Simpson  ORG
571                    Jobs, Vermilion  ORG
575  Stanford Social Innovation Review  ORG
584          Pirates of Silicon Valley  ORG
585                            Machine  ORG

[153 rows x 2 columns]

LOCATIONS
      